In [2]:
import pandas as pd

# === Load data ===
games = pd.read_csv('../NBAdata/Games.csv')
team_histories = pd.read_csv('../NBAdata/TeamHistories.csv')

# === STEP 1: CHECK TEAM IDS ===
games['winner'] = games['winner'].astype(str).str.upper()
games['hometeamId'] = games['hometeamId'].astype(str).str.upper()
games['awayteamId'] = games['awayteamId'].astype(str).str.upper()

# === STEP 2: VERIFY WINNER IS HOME OR AWAY TEAM ===
bad_rows = games[
    (games['winner'] != games['hometeamId']) &
    (games['winner'] != games['awayteamId'])
]
print("✅ Team ID mismatch rows:", bad_rows.shape[0])  # Should be 0

# === STEP 3: CONVERT AND SORT DATES ===
games['gameDate'] = pd.to_datetime(games['gameDate'], errors='coerce')
games = games.sort_values(by='gameDate').reset_index(drop=True)

# === STEP 4a: HOME WIN FLAG ===
games['home_win'] = (games['winner'] == games['hometeamId']).astype(int)

# === STEP 4b: ROLLING HOME WIN RATE (LAST 5 HOME GAMES) ===
games['rolling_home_win_rate_5'] = (
    games.groupby('hometeamId')['home_win']
    .transform(lambda x: x.shift().rolling(window=5, min_periods=1).mean())
)

# === Add rolling home win rate (last 10 home games) ===
games['rolling_home_win_rate_10'] = (
    games.groupby('hometeamId')['home_win']
    .transform(lambda x: x.shift().rolling(window=10, min_periods=1).mean())
)

# === STEP 4c: HOME WIN STREAK ===
def compute_streaks(results):
    streaks = []
    streak = 0
    for result in results:
        if result == 1:
            streak = streak + 1 if streak >= 0 else 1
        else:
            streak = streak - 1 if streak <= 0 else -1
        streaks.append(streak)
    return streaks

games['home_win_streak'] = (
    games.sort_values('gameDate')
         .groupby('hometeamId')['home_win']
         .transform(compute_streaks)
)

# === STEP 4d: MERGE LATEST TEAM NAME ===
team_histories['teamId'] = team_histories['teamId'].astype(str)
team_names = (
    team_histories
    .sort_values(by='seasonActiveTill', ascending=False)
    .drop_duplicates(subset='teamId')[['teamId', 'teamName']]
)
games = games.drop(columns=[col for col in games.columns if col.startswith('hometeamName')], errors='ignore')
games = games.merge(team_names, left_on='hometeamId', right_on='teamId', how='left')
games = games.rename(columns={'teamName': 'hometeamName'})
games = games.drop(columns=['teamId'])

# === STEP 4e: REMOVE DUPLICATE GAMES ===
games = games.drop_duplicates(subset='gameId').reset_index(drop=True)

# === STEP 5: OVERALL WIN RESULTS (HOME + AWAY COMBINED) ===
games['home_team_win'] = (games['winner'] == games['hometeamId']).astype(int)
games['away_team_win'] = (games['winner'] == games['awayteamId']).astype(int)

# === NEW FEATURE: POINT DIFFERENTIAL ===
games['home_point_diff'] = games['homeScore'] - games['awayScore']
games['away_point_diff'] = games['awayScore'] - games['homeScore']

# === NEW FEATURE: TOTAL POINTS ===
games['total_points'] = games['homeScore'] + games['awayScore']

# === NEW FEATURE: GAME TYPE (REGULAR / PLAYOFF) ===
# Extract game type from gameLabel field
games['game_type'] = games['gameType']
games['is_playoff'] = games['gameType'].str.lower().str.contains('playoff').astype(int)
games['playoff_round'] = games['gameLabel'].fillna('')

# === NEW FEATURE: SPREAD ===
games['spread'] = abs(games['homeScore'] - games['awayScore'])

# === Create team results dataframe for both home and away teams ===
home_df = games[['gameId', 'gameDate', 'hometeamId', 'home_team_win', 
                 'homeScore', 'awayScore', 'home_point_diff', 'is_playoff']].copy()
home_df = home_df.rename(columns={
    'hometeamId': 'teamId', 
    'home_team_win': 'win',
    'homeScore': 'points_scored',
    'awayScore': 'points_allowed',
    'home_point_diff': 'point_diff'
})

away_df = games[['gameId', 'gameDate', 'awayteamId', 'away_team_win', 
                 'awayScore', 'homeScore', 'away_point_diff', 'is_playoff']].copy()
away_df = away_df.rename(columns={
    'awayteamId': 'teamId', 
    'away_team_win': 'win',
    'awayScore': 'points_scored',
    'homeScore': 'points_allowed',
    'away_point_diff': 'point_diff'
})

team_game_results = pd.concat([home_df, away_df]).sort_values(by='gameDate').reset_index(drop=True)

# === STEP 5a: OVERALL ROLLING WIN RATE (5 and 10 games) ===
team_game_results['rolling_win_rate_5'] = (
    team_game_results.groupby('teamId')['win']
    .transform(lambda x: x.shift().rolling(window=5, min_periods=1).mean())
)

team_game_results['rolling_win_rate_10'] = (
    team_game_results.groupby('teamId')['win']
    .transform(lambda x: x.shift().rolling(window=10, min_periods=1).mean())
)

# === STEP 5b: OVERALL WIN STREAK ===
team_game_results['win_streak'] = (
    team_game_results
    .sort_values(by='gameDate')
    .groupby('teamId')['win']
    .transform(compute_streaks)
)

# === NEW FEATURE: ROLLING AVERAGE POINTS SCORED (5 and 10 games) ===
team_game_results['rolling_points_scored_5'] = (
    team_game_results.groupby('teamId')['points_scored']
    .transform(lambda x: x.shift().rolling(window=5, min_periods=1).mean())
)

team_game_results['rolling_points_scored_10'] = (
    team_game_results.groupby('teamId')['points_scored']
    .transform(lambda x: x.shift().rolling(window=10, min_periods=1).mean())
)

# === NEW FEATURE: ROLLING AVERAGE POINTS ALLOWED (5 and 10 games) ===
team_game_results['rolling_points_allowed_5'] = (
    team_game_results.groupby('teamId')['points_allowed']
    .transform(lambda x: x.shift().rolling(window=5, min_periods=1).mean())
)

team_game_results['rolling_points_allowed_10'] = (
    team_game_results.groupby('teamId')['points_allowed']
    .transform(lambda x: x.shift().rolling(window=10, min_periods=1).mean())
)

# === NEW FEATURE: ROLLING POINT DIFFERENTIAL AVERAGE (5 and 10 games) ===
team_game_results['rolling_point_diff_5'] = (
    team_game_results.groupby('teamId')['point_diff']
    .transform(lambda x: x.shift().rolling(window=5, min_periods=1).mean())
)

team_game_results['rolling_point_diff_10'] = (
    team_game_results.groupby('teamId')['point_diff']
    .transform(lambda x: x.shift().rolling(window=10, min_periods=1).mean())
)

# === NEW FEATURE: HOME/AWAY GAME COUNTS ===
# Create is_home column
team_game_results['is_home'] = team_game_results['teamId'].isin(
    games['hometeamId'].unique()
).astype(int)

# Add cumulative counts of home/away games per team
team_game_results['home_games_count'] = (
    team_game_results.sort_values(by='gameDate')
    .groupby('teamId')['is_home']
    .transform('cumsum')
)

team_game_results['away_games_count'] = (
    team_game_results.sort_values(by='gameDate')
    .groupby('teamId')
    .cumcount() + 1 - team_game_results['home_games_count']
)

# === Merge team games results back to main games dataframe ===
# Add home team metrics
home_metrics = team_game_results.copy()
home_metrics = home_metrics.sort_values(by='gameDate')
home_metrics_cols = [
    'gameDate', 'teamId', 'rolling_win_rate_5', 'rolling_win_rate_10', 
    'win_streak', 'rolling_points_scored_5', 'rolling_points_scored_10',
    'rolling_points_allowed_5', 'rolling_points_allowed_10',
    'rolling_point_diff_5', 'rolling_point_diff_10',
    'home_games_count', 'away_games_count'
]
home_metrics = home_metrics[home_metrics_cols]

games = games.merge(
    home_metrics,
    left_on=['gameDate', 'hometeamId'],
    right_on=['gameDate', 'teamId'],
    how='left',
    suffixes=('', '_home')
)
games = games.drop(columns=['teamId'])

# Rename columns for clarity
games = games.rename(columns={
    'rolling_win_rate_5': 'home_rolling_win_rate_5',
    'rolling_win_rate_10': 'home_rolling_win_rate_10',
    'win_streak': 'home_win_streak_overall',
    'rolling_points_scored_5': 'home_rolling_points_scored_5',
    'rolling_points_scored_10': 'home_rolling_points_scored_10',
    'rolling_points_allowed_5': 'home_rolling_points_allowed_5',
    'rolling_points_allowed_10': 'home_rolling_points_allowed_10',
    'rolling_point_diff_5': 'home_rolling_point_diff_5',
    'rolling_point_diff_10': 'home_rolling_point_diff_10',
    'home_games_count': 'home_team_home_games_count',
    'away_games_count': 'home_team_away_games_count'
})

# Add away team metrics
away_metrics = team_game_results.copy()
away_metrics = away_metrics.sort_values(by='gameDate')
away_metrics = away_metrics[home_metrics_cols]

games = games.merge(
    away_metrics,
    left_on=['gameDate', 'awayteamId'],
    right_on=['gameDate', 'teamId'],
    how='left',
    suffixes=('', '_away')
)
games = games.drop(columns=['teamId'])

# Rename columns for clarity
games = games.rename(columns={
    'rolling_win_rate_5': 'away_rolling_win_rate_5',
    'rolling_win_rate_10': 'away_rolling_win_rate_10',
    'win_streak': 'away_win_streak_overall',
    'rolling_points_scored_5': 'away_rolling_points_scored_5',
    'rolling_points_scored_10': 'away_rolling_points_scored_10',
    'rolling_points_allowed_5': 'away_rolling_points_allowed_5',
    'rolling_points_allowed_10': 'away_rolling_points_allowed_10',
    'rolling_point_diff_5': 'away_rolling_point_diff_5',
    'rolling_point_diff_10': 'away_rolling_point_diff_10',
    'home_games_count': 'away_team_home_games_count',
    'away_games_count': 'away_team_away_games_count'
})

# === NEW FEATURE: HEAD TO HEAD WIN RATE ===
# Create matchup ID for each pair of teams (alphabetically ordered to ensure consistency)
def create_matchup_id(row):
    teams = sorted([row['hometeamId'], row['awayteamId']])
    return f"{teams[0]}_{teams[1]}"

games['matchup_id'] = games.apply(create_matchup_id, axis=1)

# Determine home team win in head-to-head
games['home_win_h2h'] = games['home_win']

# Create specific matchup dataframe
matchup_results = games[['gameDate', 'matchup_id', 'hometeamId', 'awayteamId', 'home_win_h2h']].copy()

# Calculate rolling head-to-head win rates for the home team against the specific opponent
def calculate_h2h_win_rate(df):
    df = df.sort_values(by='gameDate')
    
    # For each matchup, calculate the win rate for both teams
    # First for when the team is home
    df['home_team_h2h_win_rate_10'] = (
        df.groupby(['matchup_id', 'hometeamId'])['home_win_h2h']
        .transform(lambda x: x.shift().rolling(window=10, min_periods=1).mean())
    )
    
    # Then for when the team is away (need to flip the win value)
    df['away_team_h2h_win_rate_10'] = (
        df.groupby(['matchup_id', 'awayteamId'])['home_win_h2h']
        .transform(lambda x: (1 - x).shift().rolling(window=10, min_periods=1).mean())
    )
    
    return df

matchup_results = calculate_h2h_win_rate(matchup_results)

# Merge head-to-head stats back to main games dataframe
games = games.merge(
    matchup_results[['gameDate', 'matchup_id', 'hometeamId', 'awayteamId', 
                     'home_team_h2h_win_rate_10', 'away_team_h2h_win_rate_10']],
    on=['gameDate', 'matchup_id', 'hometeamId', 'awayteamId'],
    how='left'
)

# === FINAL SORT (DESCENDING) ===
games = games.sort_values(by='gameDate', ascending=False).reset_index(drop=True)

# === SAVE PROCESSED DATA TO FILES ===
# Save the main enhanced games dataset
output_games_path = '../NBAdata/enhanced_games.csv'
games.to_csv(output_games_path, index=False)
print(f"\n✅ Enhanced games data saved to {output_games_path}")

# Save team-level game results dataset
output_team_results_path = '../NBAdata/team_game_results.csv'
team_game_results.to_csv(output_team_results_path, index=False)
print(f"✅ Team game results saved to {output_team_results_path}")

# === FINAL OUTPUT ===
print("\n✅ Enhanced Preprocessing Complete")

# Display key home team features
print("\nHome Team Features Sample:")
home_features = [
    'gameDate', 'hometeamId', 'hometeamName', 'home_win', 
    'home_rolling_win_rate_5', 'home_rolling_win_rate_10',
    'home_win_streak', 'home_win_streak_overall',
    'home_rolling_points_scored_10', 'home_rolling_points_allowed_10',
    'home_rolling_point_diff_10', 'home_team_h2h_win_rate_10'
]
print(games[home_features].head(3))

# Display key away team features
print("\nAway Team Features Sample:")
away_features = [
    'gameDate', 'awayteamId', 'awayteamName', 'away_team_win',
    'away_rolling_win_rate_5', 'away_rolling_win_rate_10',
    'away_win_streak_overall',
    'away_rolling_points_scored_10', 'away_rolling_points_allowed_10',
    'away_rolling_point_diff_10', 'away_team_h2h_win_rate_10'
]
print(games[away_features].head(3))

# Display game-specific features
print("\nGame Features Sample:")
game_features = [
    'gameDate', 'hometeamName', 'awayteamName', 
    'home_point_diff', 'total_points', 'game_type',
    'is_playoff', 'spread'
]
print(games[game_features].head(3))

# Summary of all added features
print("\n=== New Features Added ===")
print("1. Rolling win rates (5 and 10 games)")
print("2. Rolling points scored (5 and 10 games)")
print("3. Rolling points allowed (5 and 10 games)")
print("4. Rolling point differential (5 and 10 games)")
print("5. Home/away game counts")
print("6. Game type (regular/playoff)")
print("7. Point spread")
print("8. Total points")
print("9. Head-to-head win rate (last 10 matchups)")

# Data shape summary
print(f"\n=== Data Summary ===")
print(f"Enhanced games dataset: {games.shape[0]} rows × {games.shape[1]} columns")
print(f"Team game results dataset: {team_game_results.shape[0]} rows × {team_game_results.shape[1]} columns")
print(f"Files saved to ../NBAdata/ directory")

/var/folders/h5/0_hjnj_x0wj9602dnkqzthsh0000gn/T/ipykernel_24197/1462577055.py:4: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  games = pd.read_csv('../NBAdata/Games.csv')


✅ Team ID mismatch rows: 0

✅ Enhanced games data saved to ../NBAdata/enhanced_games.csv
✅ Team game results saved to ../NBAdata/team_game_results.csv

✅ Enhanced Preprocessing Complete

Home Team Features Sample:
             gameDate  hometeamId hometeamName  home_win  \
0 2025-05-05 21:30:00  1610612760      Thunder         0   
1 2025-05-05 19:00:00  1610612738      Celtics         0   
2 2025-05-04 20:30:00  1610612745      Rockets         0   

   home_rolling_win_rate_5  home_rolling_win_rate_10  home_win_streak  \
0                      1.0                       0.8               -1   
1                      0.8                       0.8               -1   
2                      0.6                       0.4               -1   

   home_win_streak_overall  home_rolling_points_scored_10  \
0                       -1                          121.1   
1                       -1                          107.4   
2                       -1                          108.2   

   home